# Data Processing for UNSW Gear Wear Dataset

The full dataset is not public. A portion of the data has been published at Mendeley Data.

-   Journal paper 1: https://www.sciencedirect.com/science/article/pii/S0888327020306440#f0040
-   Journal paper 2: https://linkinghub.elsevier.com/retrieve/pii/S0888327020304568
-   Journal paper 3: https://www.sciencedirect.com/science/article/pii/S0043164821001137
-   Journal paper 4: https://www.sciencedirect.com/science/article/pii/S0888327021006798
-   Mendeley data (partial data & data description): https://data.mendeley.com/datasets/p2yryg9k6z/2

**ERRORS**

-   Weird super higher amplitudes 34h 08m measurements


In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from signal_analysis import FFT_TSA, order_track_FFT, order_track_signal, signal_TSA
import matplotlib.pyplot as plt
from nptdms import TdmsFile

In [ ]:
sampling_rate = 100000  # Hz
pinion_teeth = 19

# OT
# < 5000, which is the number of samples per rev at 20 Hz speed
num_samples_per_revolution = pinion_teeth * 100  # 1900

# TSA
revolutions_per_TSA = 15  # ! 10 for 6 Hz and 2 Hz speeds
TSA_step_size = 5

# Raw data
data_folder = Path("/") / "Volumes" / "T7-Aalto" / "UNSW_Gear_wear_lubricated"
files = data_folder.rglob("*.tdms")

df_rows = []
for f in files:
    fname_parts = re.split(r"-|_", f.stem)

    # No need to include trial runs
    if f.parts[-2] == "Trial":
        continue

    print(f.parts)
    print(fname_parts)

    time = fname_parts[2].split("h")
    time = float(f"{time[0]}.{time[1].split('m')[0]}")
    speed = int(fname_parts[0].replace("Hz", ""))
    load = int(fname_parts[1].replace("Nm", ""))

    # ! Notice!!!
    # NOTE: Something weird happens at 34h08m, so we skip it
    if time > 34.07 and time < 34.09:
        print(
            f"Skipping 34h08m because of a weird spike in amplitudes for most orders!"
        )
        continue

    tdms_group = TdmsFile.read(f)["DAQ"]
    tdms_group.channels()

    tacho = tdms_group["SlipringTacIn"][:]
    vib1 = tdms_group["B&K4396"][:]
    vib2 = tdms_group["B&K4394"][:]

    # Tacho
    cleaned_pulses = (tacho > 3) * 1  # clean (* 1 to ensure int, not boolean)
    crossing_idxs = np.where(np.diff(cleaned_pulses) == 1)[0]

    # Check if real speed matches the one in the filename
    rot_freq = sampling_rate / np.diff(crossing_idxs)
    wrong_speed_mask = ~(rot_freq < speed + 0.2) & (rot_freq > speed - 0.2)
    if np.sum(wrong_speed_mask) > 0:
        print(rot_freq)
        print(f"Speed does not match ({speed} vs {np.mean(rot_freq)}) in {f.parts}")

    angle = np.arange(len(crossing_idxs)) * 360
    # ! Endings are incorrect because of extrapolation, but they get cut out
    interpolated_angle = np.interp(np.arange(len(tacho)), crossing_idxs, angle)

    # Signal OT
    ##

    # OT
    vib1_OT_signal = order_track_signal(
        vib1,
        np.arange(len(vib1)) / sampling_rate,
        interpolated_angle,
        num_samples_per_revolution=num_samples_per_revolution,
    )
    vib2_OT_signal = order_track_signal(
        vib2,
        np.arange(len(vib2)) / sampling_rate,
        interpolated_angle,
        num_samples_per_revolution=num_samples_per_revolution,
    )

    # Cut ends (one full revolution on both sides) because of extrapolation
    vib1_OT_signal = vib1_OT_signal[
        1 * num_samples_per_revolution : -1 * num_samples_per_revolution
    ]
    vib2_OT_signal = vib2_OT_signal[
        1 * num_samples_per_revolution : -1 * num_samples_per_revolution
    ]

    # TSA
    vib1_OT_TSA = signal_TSA(
        vib1_OT_signal,
        num_samples_per_revolution,
        (
            revolutions_per_TSA if speed > 7 else 10
        ),  # * Not enough revolutions at low speeds
        TSA_step_size,
    )
    vib2_OT_TSA = signal_TSA(
        vib2_OT_signal,
        num_samples_per_revolution,
        (
            revolutions_per_TSA if speed > 7 else 10
        ),  # * Not enough revolutions at low speeds
        TSA_step_size,
    )

    # FFT
    vib1_signal_OT_samples = np.abs(np.fft.rfft(vib1_OT_TSA, norm="forward", axis=1))[
        :, : 11 * pinion_teeth
    ]
    vib2_signal_OT_samples = np.abs(np.fft.rfft(vib2_OT_TSA, norm="forward", axis=1))[
        :, : 11 * pinion_teeth
    ]

    # Check that all sensors have the same number of samples
    # If not, something is probably wrong
    vib1_signal_OT_samples = vib1_signal_OT_samples.astype(np.float32)
    vib2_signal_OT_samples = vib2_signal_OT_samples.astype(np.float32)

    if vib1_signal_OT_samples.shape[0] != vib2_signal_OT_samples.shape[0]:
        raise ValueError("Different numbers of signal OT samples for vib1 and vib2.")

    # Explode rows
    for i in range(vib1_signal_OT_samples.shape[0]):
        df_rows.append(
            {
                "speed": speed,
                "load": load,
                "time": time,
                "OT_method": "signal",
                "vib1": vib1_signal_OT_samples[i],
                "vib2": vib2_signal_OT_samples[i],
            }
        )

    # FFT OT
    ##

    # OT & FFT
    vib1_OT_FFT = order_track_FFT(
        vib1, None, interpolated_angle, num_orders=pinion_teeth * 11
    )
    vib2_OT_FFT = order_track_FFT(
        vib2, None, interpolated_angle, num_orders=pinion_teeth * 11
    )

    # Cut ends (one full revolution on both sides) because of extrapolation
    vib1_OT_FFT = vib1_OT_FFT[1:-1]
    vib2_OT_FFT = vib2_OT_FFT[1:-1]

    # TSA
    vib1_fft_OT_samples = FFT_TSA(
        vib1_OT_FFT,
        (
            revolutions_per_TSA if speed > 7 else 10
        ),  # * Not enough revolutions at low speeds
        TSA_step_size,
    )
    vib2_fft_OT_samples = FFT_TSA(
        vib2_OT_FFT,
        (
            revolutions_per_TSA if speed > 7 else 10
        ),  # * Not enough revolutions at low speeds
        TSA_step_size,
    )

    vib1_fft_OT_samples = vib1_fft_OT_samples.astype(np.float32)
    vib2_fft_OT_samples = vib2_fft_OT_samples.astype(np.float32)

    # Check that all sensors have the same number of samples
    # If not, something is probably wrong
    if vib1_fft_OT_samples.shape[0] != vib2_fft_OT_samples.shape[0]:
        raise ValueError("Different numbers of FFT OT samples for vib1 and vib2.")

    # Explode rows
    for i in range(vib1_fft_OT_samples.shape[0]):
        df_rows.append(
            {
                "speed": speed,
                "load": load,
                "time": time,
                "OT_method": "fft",
                "vib1": vib1_fft_OT_samples[i],
                "vib2": vib2_fft_OT_samples[i],
            }
        )

df = pd.DataFrame(df_rows)
df["time"] = df["time"].astype(np.float32)

df.head(3)

('/', 'Volumes', 'T7-Aalto', 'UNSW_Gear_wear_lubricated', 'DAY15', '36th Run', 'Nominal', '20Hz-20Nm-46h20m.tdms')
['20Hz', '20Nm', '46h20m']
('/', 'Volumes', 'T7-Aalto', 'UNSW_Gear_wear_lubricated', 'DAY15', '36th Run', 'Nominal', '20Hz-20Nm-46h35m.tdms')
['20Hz', '20Nm', '46h35m']
('/', 'Volumes', 'T7-Aalto', 'UNSW_Gear_wear_lubricated', 'DAY15', '36th Run', 'Nominal', '20Hz-20Nm-46h50m.tdms')
['20Hz', '20Nm', '46h50m']
('/', 'Volumes', 'T7-Aalto', 'UNSW_Gear_wear_lubricated', 'DAY15', '36th Run', 'Nominal', '20Hz-20Nm-47h05m.tdms')
['20Hz', '20Nm', '47h05m']
('/', 'Volumes', 'T7-Aalto', 'UNSW_Gear_wear_lubricated', 'DAY15', '36th Run', 'Nominal', '20Hz-20Nm-47h20m.tdms')
['20Hz', '20Nm', '47h20m']
('/', 'Volumes', 'T7-Aalto', 'UNSW_Gear_wear_lubricated', 'DAY15', '36th Run', 'Nominal', '20Hz-20Nm-47h35m.tdms')
['20Hz', '20Nm', '47h35m']
('/', 'Volumes', 'T7-Aalto', 'UNSW_Gear_wear_lubricated', 'DAY15', '36th Run', 'Nominal', '20Hz-20Nm-47h50m.tdms')
['20Hz', '20Nm', '47h50m']
('/', 

,speed,load,time,OT_method,vib1,vib2
0,20,20,46.200001,signal,"[0.00020272976, 4.1798186e-05, 0.00011339893, ...","[3.2681066e-05, 1.3723616e-05, 1.1058246e-05, ..."
1,20,20,46.200001,signal,"[0.00018211255, 4.672557e-05, 0.00010180853, 6...","[2.8784963e-05, 1.4227893e-05, 9.093956e-06, 5..."
2,20,20,46.200001,signal,"[0.00015686985, 5.8642567e-05, 9.485911e-05, 7...","[2.8296443e-05, 1.4829064e-05, 6.5526633e-06, ..."


In [3]:
df.dtypes

speed          int64
load           int64
time         float32
OT_method     object
vib1          object
vib2          object
dtype: object

In [4]:
df.to_feather(
    Path.cwd().parent.parent
    / "data"
    # / "UNSW_gear_wear_lubricated.feather"
    / "UNSW_gear_wear_lubricated_TSA_15.feather"
)

## Testing

### Test that speeds correct


In [ ]:
tdms_sample_file = (
    Path("/")
    / "Volumes"
    / "T7-Aalto"
    / "UNSW_Gear_wear_lubricated"
    / "DAY2"
    / "2nd Run"
    / "Nominal"
    / "20Hz-20Nm-00h18m.tdms"
)

tdms_group = TdmsFile.read(tdms_sample_file)["DAQ"]
tdms_group.channels()

# signal = tdms_group["Torque"][:1000]
# signal = tdms_group["B&K4396"][:]
signal = tdms_group["SlipringTacIn"][:]
# signal = tdms_group["AE VS375"][:]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(signal)
tdms_sample_file = (
    Path("/")
    / "Volumes"
    / "T7-Aalto"
    / "UNSW_Gear_wear_lubricated"
    / "DAY15"
    / "37th Run"
    / "TE"
    / "2Hz-0Nm-50h51m.tdms"
    # / "10Hz-20Nm-00h33m (Check Speed).tdms"
)

tdms_group = TdmsFile.read(tdms_sample_file)["DAQ"]
tdms_group.channels()

# signal = tdms_group["Torque"][:1000]
# signal = tdms_group["B&K4396"][:]
signal = tdms_group["SlipringTacIn"][:]
# signal = tdms_group["AE VS375"][:]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(signal)
cleaned_pulses = (signal > 3) * 1  # clean (* 1 to ensure int, not boolean)
crossing_idxs = np.where(np.diff(cleaned_pulses) == 1)[0]
hz = 100000 / np.diff(crossing_idxs)
print(hz)
print(np.sum(hz < 1.5))
print(np.mean(hz))

# hz[hz ]

### General testing


In [ ]:
df_testing = pd.read_feather(
    Path.cwd().parent.parent / "data" / "UNSW_gear_wear_lubricated.feather"
)

df_testing.sort_values(
    by=["time", "speed", "load", "OT_method"],
    inplace=True,
)
df_testing.reset_index(drop=True, inplace=True)

df_testing.groupby(["speed", "load", "OT_method"]).count()

In [ ]:
# Most measurements are at 20 Hz and 20 Nm, so we can use that as a nominal case

nominal = df_testing[
    (df_testing["speed"] == 20)
    & (df_testing["load"] == 20)
    & (df_testing["OT_method"] == "signal")
]

px.bar(np.log(nominal.iloc[-1]["vib1"]) - np.log(nominal.iloc[0]["vib1"]))

In [ ]:
x1 = np.array([x for x in nominal["vib1"]])[:, 18]
x12 = np.array([x for x in nominal["vib1"]])[:, 19]
x13 = np.array([x for x in nominal["vib1"]])[:, 17]

fig = go.Figure()
fig.add_trace(go.Scatter(y=x1, mode="lines", name="x1"))
fig.add_trace(go.Scatter(y=x12, mode="lines", name="x1 + 1"))
fig.add_trace(go.Scatter(y=x13, mode="lines", name="x1 - 1"))
fig.update_layout(
    title="1 x GMFO and closest sidebands",
    xaxis_title="Sample",
    yaxis_title="Amplitude",
)
fig.show()

In [ ]:
x2 = np.array([x for x in nominal["vib1"]])[:, 37]
x22 = np.array([x for x in nominal["vib1"]])[:, 38]
x23 = np.array([x for x in nominal["vib1"]])[:, 36]

fig = go.Figure()
fig.add_trace(go.Scatter(y=x2, mode="lines", name="x2"))
fig.add_trace(go.Scatter(y=x22, mode="lines", name="x2 + 1"))
fig.add_trace(go.Scatter(y=x23, mode="lines", name="x2 - 1"))
fig.update_layout(
    title="2 x GMFO and closest sidebands",
    xaxis_title="Sample",
    yaxis_title="Amplitude",
)
fig.show()
# plt.plot(x2)
# plt.plot(x22)
# plt.plot(x23)

In [ ]:
x3 = np.array([x for x in nominal["vib1"]])[:, 56]
x32 = np.array([x for x in nominal["vib1"]])[:, 57]
x33 = np.array([x for x in nominal["vib1"]])[:, 55]
fig = go.Figure()
fig.add_trace(go.Scatter(y=x3, mode="lines", name="x3"))
fig.add_trace(go.Scatter(y=x32, mode="lines", name="x3 + 1"))
fig.add_trace(go.Scatter(y=x33, mode="lines", name="x3 - 1"))
fig.update_layout(
    title="3 x GMFO and closest sidebands",
    xaxis_title="Sample",
    yaxis_title="Amplitude",
)
fig.show()
# plt.plot(x3)
# plt.plot(x32)
# plt.plot(x33)

In [ ]:
time = nominal["time"].to_numpy()
time = time + np.arange(len(time))  # * 0.01  # Add a small offset for visualization
x3 = np.array([x for x in nominal["vib1"]])[:, 18]
fig = go.Figure()
fig.add_trace(go.Scatter(x=time, y=x3, mode="lines", name="x3"))
fig.update_layout(title="", xaxis_title="Time", yaxis_title="Amplitude")
fig.show()